# Assignment 1: Housing Price Prediction

**Goal:** Predict `SalePrice` for houses in the Ames Housing dataset using a complete machine-learning workflow:

**Explore data → Split data → Preprocess → Select/engineer features → Train models → Evaluate and interpret**

The focus is not the highest possible score, but understanding the data and being able to **explain and justify** your decisions. Write short explanations in the markdown cells along the way (replace the text marked *✏️ Your answer*).

> Use only `AmesHousing.csv`, and the *Ames Housing – Understanding the Dataset* guide whenever a column name or code is unclear.

## 0. Setup

Import the libraries you need. This cell is already filled in – add more imports if needed.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_squared_error, r2_score

RANDOM_STATE = 42          # used everywhere for reproducible results
pd.set_option("display.max_columns", 100)

### Load the dataset

In [ ]:
df = pd.read_csv("AmesHousing.csv")
print(df.shape)
df.head()

---
## Step 1 – Explore and understand the data

Explore the dataset using descriptive statistics and visualizations. Choose analyses that actually help you make later decisions – do not create plots just for the sake of it.

The assignment asks you to investigate:
1. the available variables and their data types
2. the distribution of `SalePrice`
3. missing values
4. unusual values or patterns
5. relationships that may be useful for predicting house prices

### 1.1 Variables and data types

In [ ]:
df.info()
df.dtypes


In [ ]:
num_cols = df.select_dtypes(include="number").columns
print(num_cols)
cat_cols = df.select_dtypes(exclude="number").columns
print(cat_cols)

*✏️ What kinds of variables do we have?*

* 2930 houses and 82 columns: 39 numerical and 43 categorical. The target is `SalePrice`.
* `Order` and `PID` are only IDs. `MS SubClass` is stored as a number but is really a category code.

### 1.2 Distribution of `SalePrice`

In [ ]:
print(df["SalePrice"].describe())
print("Skewness:", df["SalePrice"].skew())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["SalePrice"], kde=True, ax=axes[0])
axes[0].set_title("SalePrice")
sns.histplot(np.log1p(df["SalePrice"]), kde=True, ax=axes[1])
axes[1].set_title("log(1 + SalePrice)")
plt.show()

*✏️ What does the distribution of `SalePrice` look like?*

* Right skewed with skewness 1.74. Most houses sell for 130k to 214k, but a long tail goes up to 755k and pulls the mean above the median.
* A log transform makes it almost symmetric, which could help Linear Regression.

### 1.3 Missing values

In [ ]:
missing_count = df.isnull().sum()                 # antall NaN per kolonne
missing_pct = missing_count / len(df) * 100       # andel i prosent

missing = pd.DataFrame({
    "Missing": missing_count,
    "Percent": missing_pct.round(2)
})

missing = missing[missing["Missing"] > 0].sort_values("Missing", ascending=False)
print(f"{len(missing)} columns have missing values")
missing

*✏️ Which columns have many missing values?*

* `Pool QC`, `Misc Feature`, `Alley` and `Fence` are over 80 % missing.
* According to the guide, missing here usually means the house does not have the feature, so it is real information and not an error.

### 1.4 Unusual values and patterns

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(x="Gr Liv Area", y="SalePrice", hue="Sale Condition", data=df, alpha=0.6)
plt.axvline(4000, color="red", linestyle="--")
plt.title("Living area vs SalePrice")
plt.show()

df[df["Gr Liv Area"] > 4000][["Gr Liv Area", "SalePrice", "Sale Condition", "Neighborhood", "Overall Qual"]]

In [ ]:
df.describe().T[["min", "max"]]
print(df[df["Garage Yr Blt"] > df["Yr Sold"]][["Year Built", "Garage Yr Blt", "Yr Sold"]])
print(df[df["Year Remod/Add"] < df["Year Built"]][["Year Built", "Year Remod/Add"]])
print(df[df["Yr Sold"] < df["Year Built"]][["Year Built", "Yr Sold"]])

In [ ]:
# Share of the most frequent value in each categorical column
top_share = df[cat_cols].apply(lambda c: c.value_counts(normalize=True, dropna=False).iloc[0])
top_share.sort_values(ascending=False).head(10)

*✏️ Which unusual observations did we find?*

* Three very large houses in Edwards sold for only 160k to 185k. They are partial sales and do not follow the trend.
* One garage is built in 2207, which is a typing error.
* Columns like `Utilities` and `Street` have the same value for over 98 % of the houses, so they carry almost no information.

### 1.5 Relationships with `SalePrice`

In [ ]:
corr = df.drop(columns=["Order", "PID"]).corr(numeric_only=True)
corr["SalePrice"].sort_values(ascending=False)

In [ ]:
top10 = corr["SalePrice"].abs().sort_values(ascending=False).index[:11]   # SalePrice + top 10

plt.figure(figsize=(10, 8))
sns.heatmap(corr.loc[top10, top10], annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation between the top features")
plt.show()

In [ ]:
features = ["Overall Qual", "Gr Liv Area", "Total Bsmt SF", "Year Built"]

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, col in zip(axes, features):
    sns.scatterplot(x=col, y="SalePrice", data=df, alpha=0.4, ax=ax)
    ax.set_title(f"{col} vs SalePrice")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, ["Exter Qual", "Kitchen Qual", "Bsmt Qual"]):
    order = df.groupby(col)["SalePrice"].median().sort_values().index
    sns.boxplot(x=col, y="SalePrice", data=df, order=order, ax=ax)
plt.tight_layout()
plt.show()

# Neighborhood has many categories – give it its own large plot
order = df.groupby("Neighborhood")["SalePrice"].median().sort_values().index
plt.figure(figsize=(14, 5))
sns.boxplot(x="Neighborhood", y="SalePrice", data=df, order=order)
plt.xticks(rotation=90)
plt.title("SalePrice by Neighborhood")
plt.show()

*✏️ Which variables are most related to price? (→ Q1, Q2)*

* Price mainly depends on quality, size and age. The strongest are `Overall Qual` 0.80, `Gr Liv Area` 0.71, `Garage Cars` 0.65 and `Total Bsmt SF` 0.63.
* `Neighborhood` and the quality ratings show big price differences in the box plots.
* Some features are almost duplicates, like `Garage Cars` and `Garage Area`, so we do not need both.

---
## Step 2 – Create training and test data

Split the dataset into a training set and a test set. **All** further development (preprocessing, feature selection) must be done on the training data. The test set is only used at the very end.

In [ ]:
X = df.drop(columns=["SalePrice"])
y = df["SalePrice"]

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size= 0.2,
    random_state= RANDOM_STATE,
)

print(X_train.shape)
print(X_test.shape)


*✏️ How did we split the data? (→ Q3)*

* 80 % training and 20 % test with `random_state=42`, so the split is the same every time.
* The test set is only used at the very end. This way it acts like new houses the model has never seen, and gives an honest score.

---
## Step 3 – Prepare the data

Decide how to prepare the data for the models. **Important:** preprocessing must be *learned* (fit) on the training data and then *applied* (transform) in the same way to the test data.

Possible steps:
- handle missing values (based on what they mean in the guide)
- encode categorical variables
- scale/transform numerical variables where appropriate
- remove variables when there is a clear reason to do so

### 3.1 Remove irrelevant columns

In [ ]:
# ID columns – unique for every house, contain no information about the price
id_cols = ["Order", "PID"]

# Share of the most frequent value in each column – computed on the TRAINING data only
top_share = X_train.apply(lambda c: c.value_counts(normalize=True, dropna=False).iloc[0])
print(top_share.sort_values(ascending=False).head(12).round(3))

# 98 % of the training houses have the same value
near_constant = top_share[top_share >= 0.98].index.tolist()

# Misc values have to value in 96%
misc_cols = ["Misc Feature", "Misc Val"]

drop_cols = id_cols + near_constant + misc_cols
print("Dropping", len(drop_cols), "columns:", drop_cols)

# Ignore errors
X_train = X_train.drop(columns=drop_cols, errors="ignore")
X_test = X_test.drop(columns=drop_cols, errors="ignore")
print("X_train:", X_train.shape, " X_test:", X_test.shape)

**Removed columns and why**

* `Order` and `PID` are only IDs.
* `Utilities`, `Street`, `Condition 2`, `Roof Matl`, `Heating`, `Low Qual Fin SF` and `3Ssn Porch` are almost constant, over 98 % the same value.
* Pool and misc columns are almost always empty and have no link to price.

### 3.2 Missing values that mean "does not exist"

In [ ]:
def fill_absent_features(X):
    """Fill missing values that mean "the house does not have this feature" (see the dataset guide).
    Rule-based (not learned from data), so it is safe to apply to both X_train and X_test."""
    X = X.copy()

    # 1) Missing always means "does not exist"
    for col in ["Alley", "Fence", "Fireplace Qu"]:
        X[col] = X[col].fillna("None")

    # 2) Garage: 'Garage Type' missing = no garage (all these houses have Garage Area = 0).
    #    Houses that DO have a garage but a missing value keep NaN -> "not recorded", imputed in 3.3.
    no_garage = X["Garage Type"].isna()
    for col in ["Garage Type", "Garage Finish", "Garage Qual", "Garage Cond"]:
        X.loc[no_garage, col] = "None"

    # 3) Basement: no basement if 'Bsmt Qual' is missing and the total basement area is 0 (or missing)
    no_bsmt = X["Bsmt Qual"].isna() & X["Total Bsmt SF"].fillna(0).eq(0)
    for col in ["Bsmt Qual", "Bsmt Cond", "Bsmt Exposure", "BsmtFin Type 1", "BsmtFin Type 2"]:
        X.loc[no_bsmt, col] = "None"
    for col in ["BsmtFin SF 1", "BsmtFin SF 2", "Bsmt Unf SF", "Total Bsmt SF",
                "Bsmt Full Bath", "Bsmt Half Bath"]:
        X.loc[no_bsmt, col] = X.loc[no_bsmt, col].fillna(0)

    # 4) Masonry veneer: type missing and area = 0 -> no veneer.
    #    Type missing but area > 0 (or missing) -> "not recorded", imputed in 3.3.
    no_veneer = X["Mas Vnr Type"].isna() & X["Mas Vnr Area"].eq(0)
    X.loc[no_veneer, "Mas Vnr Type"] = "None"

    return X


X_train = fill_absent_features(X_train)
X_test = fill_absent_features(X_test)

# Remaining missing values in the training data = "not recorded" -> handled by the imputers in 3.3
remaining = X_train.isnull().sum()
remaining[remaining > 0].sort_values(ascending=False)

**Missing values that mean "does not exist"**

* When the guide says missing means no alley, fence, fireplace, garage or basement, we use the category `"None"` and set related areas to 0.
* We check related columns first, for example that houses with no `Garage Type` also have `Garage Area` = 0.
* Values that are truly not recorded stay NaN and are filled in 3.3.
* These are fixed rules, so they are safe to use on both train and test.

### 3.3 Preprocessing pipeline (imputation, encoding, scaling)

In [ ]:
# 'MS SubClass' is a code for the type of dwelling (20, 60, 190, ...), not a quantity -> treat it as categorical
X_train["MS SubClass"] = X_train["MS SubClass"].astype(str)
X_test["MS SubClass"] = X_test["MS SubClass"].astype(str)


def make_preprocessor(num_features, cat_features):
    """Build a new (unfitted) preprocessor for the given feature lists.
    Used again in Step 4/5 when the final feature lists are chosen."""
    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),         # "not recorded" numbers -> training median
        ("scaler", StandardScaler()),                          # mean 0, std 1 (mainly for Linear Regression)
    ])
    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),  # "not recorded" categories -> most common in training
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    return ColumnTransformer([
        ("num", num_pipe, num_features),
        ("cat", cat_pipe, cat_features),
    ])


# For now: use all remaining columns (the final selection is done in Step 4)
num_features = X_train.select_dtypes(include="number").columns.tolist()
cat_features = X_train.select_dtypes(exclude="number").columns.tolist()
print(len(num_features), "numerical and", len(cat_features), "categorical features")

preprocessor = make_preprocessor(num_features, cat_features)

# Fit ONLY on the training data, then apply the same learned transformation to the test data
X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)
print("X_train_prep:", X_train_prep.shape, " X_test_prep:", X_test_prep.shape)

# Example of values learned from the training data
medians = pd.Series(preprocessor.named_transformers_["num"]["imputer"].statistics_, index=num_features)
print("Median 'Lot Frontage' learned from training data:", medians["Lot Frontage"])
print("Missing values left after preprocessing:", np.isnan(X_train_prep).sum(), "(train),", np.isnan(X_test_prep).sum(), "(test)")

*✏️ Preprocessing choices (→ Q4)*

* `MS SubClass` is changed to categorical, since its numbers are codes for house types.
* Missing numbers get the training median, which is robust to skewed data. Missing categories get the most common training value.
* Categories are one-hot encoded, so they do not get a false order. Unknown test categories become zeros.
* Numbers are standardised because the columns have very different scales. This matters most for Linear Regression.
* Everything is fitted on the training data only, so no information from the test set leaks into the model.

---
## Step 4 – Select and engineer features

Decide which features to use. Use evidence from the **training data** (correlations, plots, understanding of the variables). You do not have to use every column.

*Tip: Steps 3 and 4 are connected – it may be easiest to do the feature engineering (4.1) before building the pipeline in 3.3, since the pipeline needs the final `num_features` and `cat_features` lists.*

### 4.1 Feature engineering

In [ ]:
def add_features(X):
    """Create new features from existing columns. Rule-based, so it is applied to both X_train and X_test."""
    X = X.copy()

    # Total living space incl. basement – buyers pay for the total size of the house
    X["Total SF"] = X["Total Bsmt SF"] + X["1st Flr SF"] + X["2nd Flr SF"]

    # Age of the house and years since remodelling at the time of sale (clip: one house was sold before it was finished)
    X["House Age"] = (X["Yr Sold"] - X["Year Built"]).clip(lower=0)
    X["Remod Age"] = (X["Yr Sold"] - X["Year Remod/Add"]).clip(lower=0)

    # Total number of bathrooms (a half bath counts as 0.5)
    X["Total Bath"] = (X["Full Bath"] + 0.5 * X["Half Bath"]
                       + X["Bsmt Full Bath"] + 0.5 * X["Bsmt Half Bath"])

    # Total outdoor area (deck + porches)
    X["Total Porch SF"] = (X["Wood Deck SF"] + X["Open Porch SF"]
                           + X["Enclosed Porch"] + X["Screen Porch"])

    # Has the house been remodelled? (Year Remod/Add equals Year Built if not)
    X["Remodeled"] = (X["Year Remod/Add"] != X["Year Built"]).astype(int)

    return X


X_train = add_features(X_train)
X_test = add_features(X_test)

new_features = ["Total SF", "House Age", "Remod Age", "Total Bath", "Total Porch SF", "Remodeled"]

In [ ]:
# Correlation with SalePrice – computed on the TRAINING data only
compare = ["Total SF", "Gr Liv Area", "Total Bsmt SF", "1st Flr SF",
           "House Age", "Year Built", "Remod Age", "Year Remod/Add",
           "Total Bath", "Full Bath", "Total Porch SF", "Wood Deck SF", "Open Porch SF", "Remodeled"]
X_train[compare].corrwith(y_train).round(3).to_frame("corr with SalePrice")

**New features**

* `Total SF` is the total area of all floors. Correlation 0.78, higher than any single area column.
* `Total Bath` counts all bathrooms, with half baths as 0.5. Correlation 0.64, higher than `Full Bath` alone.
* `Total Porch SF` combines deck and porch areas. `House Age` and `Remod Age` replace the build and remodel years.
* `Remodeled` turned out to be almost useless with correlation −0.06.

### 4.2 Feature selection

In [ ]:
# --- Numerical features: correlation with SalePrice (TRAINING data only) ---
num_all = X_train.select_dtypes(include="number").columns
num_corr = X_train[num_all].corrwith(y_train).sort_values(key=abs, ascending=False)

# 1) Keep features with at least a moderate linear relationship with price
CORR_THRESHOLD = 0.3
candidates = num_corr[num_corr.abs() >= CORR_THRESHOLD].index.tolist()

# 2) Remove redundancy: go through the candidates from strongest to weakest, and skip a feature
#    if it is almost a duplicate (|r| > 0.8) of a feature that is already selected
REDUNDANCY_THRESHOLD = 0.8
feature_corr = X_train[candidates].corr().abs()
num_features, removed = [], {}
for col in candidates:
    duplicate_of = [kept for kept in num_features if feature_corr.loc[col, kept] > REDUNDANCY_THRESHOLD]
    if duplicate_of:
        removed[col] = f"r = {feature_corr.loc[col, duplicate_of[0]]:.2f} with '{duplicate_of[0]}'"
    else:
        num_features.append(col)

print(f"{len(candidates)} of {len(num_all)} numerical features have |r| >= {CORR_THRESHOLD}")
print("\nRemoved as redundant:")
for col, reason in removed.items():
    print(f"  {col:15s} {reason}")
print(f"\nSelected numerical features ({len(num_features)}):")
num_corr[num_features].round(3).to_frame("corr with SalePrice")

In [ ]:
# --- Categorical features: how much of the variation in SalePrice is explained by the category? ---
# Correlation ratio (eta²): 0 = the category says nothing about price, 1 = the category determines the price.
def eta_squared(x, y):
    group_means = y.groupby(x.fillna("Missing")).transform("mean")
    return ((group_means - y.mean()) ** 2).sum() / ((y - y.mean()) ** 2).sum()

cat_all = X_train.select_dtypes(exclude="number").columns
cat_eta = pd.Series({col: eta_squared(X_train[col], y_train) for col in cat_all}).sort_values(ascending=False)

plt.figure(figsize=(8, 8))
sns.barplot(x=cat_eta.values, y=cat_eta.index)
ETA_THRESHOLD = 0.1
plt.axvline(ETA_THRESHOLD, color="red", linestyle="--")
plt.xlabel("eta² (share of SalePrice variance explained)")
plt.title("Categorical features vs SalePrice (training data)")
plt.show()

cat_features = cat_eta[cat_eta >= ETA_THRESHOLD].index.tolist()

# 'Exterior 2nd' is the same as 'Exterior 1st' for most houses (only differs if the house has two materials)
print(f"'Exterior 1st' == 'Exterior 2nd' for {(X_train['Exterior 1st'] == X_train['Exterior 2nd']).mean():.0%} of the houses")
cat_features.remove("Exterior 2nd")

print(f"\nSelected categorical features ({len(cat_features)}):")
cat_eta[cat_features].round(3).to_frame("eta²")

In [ ]:
# --- Final feature lists -> rebuild the preprocessor from 3.3 ---
print(f"Final features: {len(num_features)} numerical + {len(cat_features)} categorical")
print("Numerical:  ", num_features)
print("Categorical:", cat_features)

preprocessor = make_preprocessor(num_features, cat_features)
X_train_prep = preprocessor.fit_transform(X_train)   # fit on training data only
X_test_prep = preprocessor.transform(X_test)
print("\nX_train_prep:", X_train_prep.shape, " X_test_prep:", X_test_prep.shape)

**How the features were selected**

* Numerical: keep features with |r| ≥ 0.3 with price, then drop near duplicates with |r| > 0.8. For example `Gr Liv Area` is dropped because of `Total SF`, and `Year Built` because of `House Age`. This gives 15 features.
* Categorical: keep features with eta² ≥ 0.1. `Neighborhood` is the strongest. `Exterior 2nd` is dropped because it equals `Exterior 1st` for 85 % of the houses. This gives 17 features.

*✏️ Which features did we finally use? (→ Q2, Q5)*

* 15 numerical and 17 categorical features, which gives 157 columns after one-hot encoding instead of 289.
* All three models use the same features, so they can be compared fairly.

---
## Step 5 – Train the models

Train the three models **using the same prepared features**, so that they can be compared fairly. Use the default hyperparameters (no tuning required).

- `LinearRegression`
- `DecisionTreeRegressor`
- `RandomForestRegressor`

In [ ]:
# HINT: Create a dictionary with the models:
#   models = {
#       "Linear Regression": LinearRegression(),
#       "Decision Tree Regression": DecisionTreeRegressor(random_state=RANDOM_STATE),
#       "Random Forest Regression": RandomForestRegressor(random_state=RANDOM_STATE),
#   }
# Loop over them, build a Pipeline([("prep", preprocessor), ("model", model)]) for each,
# and call .fit(X_train, y_train). Store the trained pipelines in a new dictionary.

---
## Step 6 – Evaluate and interpret the results

Evaluate all three models on the **test data** using:
- **RMSE** – the size of the prediction errors in the same units as `SalePrice` (lower is better)
- **R²** – compares the predictions with a baseline that always predicts the mean (higher is generally better; 1 = perfect, 0 = no better than the mean, can be negative). *Do not describe R² as percentage accuracy.*

| Model | Test RMSE | Test R² |
|---|---|---|
| Linear Regression | | |
| Decision Tree Regression | | |
| Random Forest Regression | | |

In [ ]:
# HINT: For each trained model:
#   y_pred = model.predict(X_test)
#   rmse = np.sqrt(mean_squared_error(y_test, y_pred))
#   r2   = r2_score(y_test, y_pred)
# Collect the results in a pd.DataFrame with the columns "Model", "Test RMSE", "Test R²".
# If you log-transformed y: remember to transform the predictions back (np.expm1) before computing RMSE!

### 6.1 Visualizing the predictions

In [ ]:
# HINT: Scatter plot of y_test (x-axis) vs y_pred (y-axis) for each model,
# with a diagonal line y = x for perfect predictions. Consider plt.subplots(1, 3).
# Look for where the models make the largest errors (expensive houses? cheap houses?).

*✏️ Your answer: Which model performed best on the test data? Why do you think so? (→ Q6)*

---
## Summary and possible improvements

*✏️ Your answer: What could you change in the preprocessing or features to improve the results without changing to a different model? (→ Q7)*